# 30. HS6 Class-Balanced Bootstrap

29번 실험의 통합 데이터·중복 제거·HS6 적격 기준·제품군 정의·LOPO 평가 구조를 그대로 재사용합니다.

- Fold별 학습 데이터만 HS6 클래스당 90건으로 Bootstrap
- Bootstrap 10회 반복
- 평가 Fold는 원본 그대로 유지
- SBERT Product + Ruling 모델 유지
- Macro-F1 우선, HS4 Accuracy·HS6 Accuracy·Weighted-F1·Top-3 및 클래스별 성능 비교

아래 셀 하나로 실행하며 결과 ZIP을 생성합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

import json, requests
from IPython.display import display

SOURCE_URL = "https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/29_discriminative_features_experiment.ipynb"
source_nb = requests.get(SOURCE_URL).json()
cells = source_nb["cells"]
for i in [3,5,7,9,11]:
    exec("".join(cells[i]["source"]), globals())
print("기준 데이터:", len(eval_df), "| HS6:", len(labels), "| 제품군:", eval_df["product_family"].nunique())

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from pathlib import Path
import zipfile

TARGET_COUNT = 90
N_BOOTSTRAPS = 10
RANDOM_STATE = 42

def bootstrap_balance_indices(train_df, target_count=90, random_state=42):
    rng = np.random.default_rng(random_state)
    selected = []
    for label, group in train_df.groupby("hs6", sort=True):
        idx = group.index.to_numpy()
        selected.extend(rng.choice(idx, size=target_count, replace=len(idx) < target_count).tolist())
    rng.shuffle(selected)
    return np.asarray(selected, dtype=int)

logo = LeaveOneGroupOut()
splits = list(logo.split(eval_df, eval_df["hs6"], groups=eval_df["product_family"]))
prediction_rows, fold_rows = [], []

for fold, (tr, te) in enumerate(splits, 1):
    train, test = eval_df.iloc[tr].copy(), eval_df.iloc[te].copy()
    if not set(test["hs6"]).issubset(set(train["hs6"])):
        raise ValueError(f"Fold {fold}: 학습 데이터에 없는 HS6 코드")
    family = test["product_family"].iloc[0]
    fold_rows.append({"fold": fold, "heldout_family": family, "train_n_original": len(train), "test_n": len(test)})
    for bootstrap_id in range(1, N_BOOTSTRAPS + 1):
        sampled_idx = bootstrap_balance_indices(train, TARGET_COUNT, RANDOM_STATE + fold * 1000 + bootstrap_id)
        sampled_codes = eval_df.loc[sampled_idx, "hs6"].to_numpy()
        for row_idx in te:
            rank = rank_case_codes(ruling_emb[sampled_idx] @ query_emb[row_idx], sampled_codes)
            prediction_rows.append({
                "row_index": int(row_idx), "fold": fold, "bootstrap": bootstrap_id,
                "heldout_family": family, "data_source": eval_df.loc[row_idx, "data_source"],
                "true_hs6": eval_df.loc[row_idx, "hs6"], "true_hs4": eval_df.loc[row_idx, "hs4"],
                "predicted_hs6": str(rank.index[0]), "top3_candidates": top_codes(rank)
            })

predictions = pd.DataFrame(prediction_rows)
fold_audit = pd.DataFrame(fold_rows)

def metric_summary(part):
    yt, yp = part["true_hs6"].astype(str), part["predicted_hs6"].astype(str)
    top3 = np.mean([str(t) in [str(x) for x in c] for t, c in zip(yt, part["top3_candidates"])])
    return {
        "n": len(part),
        "hs4_accuracy": np.mean(yt.str[:4].to_numpy() == yp.str[:4].to_numpy()),
        "hs6_accuracy": accuracy_score(yt, yp),
        "macro_f1_hs6": f1_score(yt, yp, labels=labels, average="macro", zero_division=0),
        "weighted_f1_hs6": f1_score(yt, yp, average="weighted", zero_division=0),
        "top3_hs6_accuracy": top3
    }

bootstrap_metrics = pd.DataFrame([
    {"bootstrap": b, **metric_summary(part)}
    for b, part in predictions.groupby("bootstrap")
])
summary = bootstrap_metrics.drop(columns="bootstrap").agg(["mean", "std", "min", "max"]).reset_index(names="statistic")

class_rows = []
for b, part in predictions.groupby("bootstrap"):
    yt, yp = part["true_hs6"].astype(str), part["predicted_hs6"].astype(str)
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=labels, zero_division=0)
    for code, pv, rv, fv, sv in zip(labels, p, r, f, s):
        class_rows.append({"bootstrap": b, "hs6": code, "n": int(sv), "precision": pv, "recall": rv, "f1": fv})
class_metrics = pd.DataFrame(class_rows)
class_summary = class_metrics.groupby("hs6").agg(
    precision_mean=("precision", "mean"), precision_std=("precision", "std"),
    recall_mean=("recall", "mean"), recall_std=("recall", "std"),
    f1_mean=("f1", "mean"), f1_std=("f1", "std"), support=("n", "first")
).reset_index()

display(bootstrap_metrics.round(4))
display(summary.round(4))
display(class_summary.sort_values("f1_mean").round(4))

per_fold_metrics = pd.DataFrame([
    {"fold": fold, "heldout_family": family, **metric_summary(part)}
    for (fold, family), part in predictions.groupby(["fold", "heldout_family"])
])
mode_predictions = predictions.groupby(["row_index", "true_hs6", "true_hs4"], as_index=False).agg(
    predicted_hs6=("predicted_hs6", lambda x: x.mode().iloc[0])
)
confusion = pd.crosstab(mode_predictions["true_hs6"], mode_predictions["predicted_hs6"])

OUT = Path("/content/hs_class_balanced_bootstrap_30")
OUT.mkdir(parents=True, exist_ok=True)
outputs = {
    "30_bootstrap_oof_predictions.csv": predictions,
    "30_bootstrap_metrics.csv": bootstrap_metrics,
    "30_metrics_summary.csv": summary,
    "30_class_metrics_by_bootstrap.csv": class_metrics,
    "30_class_metrics_summary.csv": class_summary,
    "30_fold_audit.csv": fold_audit,
    "30_fold_metrics.csv": per_fold_metrics,
    "30_confusion_matrix_mode_prediction.csv": confusion
}
for name, data in outputs.items():
    data.to_csv(OUT / name, index=False, encoding="utf-8-sig")
with zipfile.ZipFile("/content/30_class_balanced_bootstrap_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob("*.csv"):
        z.write(f, arcname=f.name)
print("완료: /content/30_class_balanced_bootstrap_results.zip")
